# OpenHPL SimpleTurbine on Railway
The container runs OpenModelica first with minimal overhead, then this notebook reads the real simulation result and plots `turbine.Wdot_s`. If the result file is missing, the notebook can invoke the same runner.


In [ ]:
import subprocess, pathlib, pandas as pd, matplotlib.pyplot as plt
RES = pathlib.Path('/workspace/results/SimpleTurbine_res.csv')
print(subprocess.check_output(['omc','--version'], text=True).strip())
if not RES.exists():
    print('Result missing: running low-memory OpenHPL runner...')
    subprocess.run(['/workspace/OpenHPL-repo/railway/run_model.sh'], check=True)
print('Result:', RES, 'exists=', RES.exists())


In [ ]:
df = pd.read_csv(RES)
print('Columns:', df.columns.tolist())
power_col = next(c for c in df.columns if 'turbine.Wdot_s' in c)
out = df[['time', power_col]].rename(columns={power_col:'turbine_Wdot_s_W'})
out['turbine_Wdot_s_MW'] = out['turbine_Wdot_s_W']/1e6
out.to_csv('/workspace/results/turbine_power.csv', index=False)
plt.figure(figsize=(9,5))
plt.plot(out['time'], out['turbine_Wdot_s_MW'])
plt.axvline(500, linestyle='--')
plt.xlabel('Time [s]')
plt.ylabel('Turbine shaft power [MW]')
plt.title('OpenHPL SimpleTurbine: turbine.Wdot_s')
plt.grid(True)
plt.tight_layout()
plt.savefig('/workspace/results/turbine_power.png', dpi=180)
plt.show()
print('POWER_DATA_START')
for _, row in out.iterrows():
    print(f"{row['time']:.6g},{row['turbine_Wdot_s_MW']:.12g}")
print('POWER_DATA_END')
for t in [0,490,500,515,530,540,1000]:
    i=(out['time']-t).abs().idxmin()
    print(f"sample t={out.loc[i,'time']:.1f}s P={out.loc[i,'turbine_Wdot_s_MW']:.6f} MW")
